# LSTM Bandwidth Demand Predictor

Predicts the next `HORIZON` bandwidth readings from the last `WINDOW` readings.

**What's improved over the first version**
1. One self-contained notebook (no external script, upload happens once, no duplicate cells).
2. Looks at the data first: the trace mixes a low regime (~0-20 Mbit/s) with spikes up to ~1000 Mbit/s.
3. Model predicts the *change from the last value* (so it can never be much worse than "repeat the last value"), uses extra inputs (change, rolling mean, rolling std) and an L1 loss (matches the MAE metric, robust to spikes).
4. Compares **window sizes** (the epoch comparison before was meaningless: early stopping ended both runs at the same point) and picks the best on **validation** data, not test data.
5. Reports errors separately for normal vs. spike readings, plus plots and a reusable `forecast()` function.

Run the cells top to bottom. Set the file path in the next cell.

In [ ]:
# ---------------- CONFIG: edit this cell ----------------
CSV_PATH   = "Dataset/Dataset_2/Dataset_2/QTrain/QtrainNew.csv"   # one bandwidth value per line, e.g. "3.96,Mbits/sec"
HORIZON    = 5            # how many future readings to predict
WINDOWS    = [10, 30, 60] # history lengths to compare
HIDDEN, LAYERS, DROPOUT = 128, 2, 0.2
LR, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 256, 60, 8
SEED       = 42
HIGH_MBPS  = 100.0        # readings at/above this are reported as "spike" readings

In [ ]:
import os, sys, copy, json, zipfile, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

IN_COLAB = "google.colab" in sys.modules

# Upload once (Colab only) if the file is not already there.
if not Path(CSV_PATH).exists():
    if not IN_COLAB:
        raise FileNotFoundError(f"{CSV_PATH} not found. Fix CSV_PATH in the config cell.")
    from google.colab import files
    print("Upload the dataset .csv (or the project .zip):")
    up = files.upload()
    for name in up:
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(name).extractall("project")
    hits = list(Path(".").rglob(Path(CSV_PATH).name))
    csvs = [n for n in up if n.lower().endswith(".csv")]
    CSV_PATH = str(hits[0]) if hits else (csvs[0] if csvs else CSV_PATH)
print("Using:", CSV_PATH)

## 1. Load the data

In [ ]:
def read_series(path):
    """One reading per line. Takes the last number on each line, so 'x,Mbits/sec',
    'time x' and a header row all work. Binary/garbage bytes are ignored."""
    vals = []
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            nums = []
            for tok in line.replace(",", " ").split():
                try: nums.append(float(tok))
                except ValueError: pass
            if nums: vals.append(nums[-1])
    return np.clip(np.asarray(vals, dtype=np.float32), 0, None)

series = read_series(CSV_PATH)
print(f"{len(series)} readings")
print(pd.Series(series).describe().round(2).to_string())

## 2. Look at the data before modelling

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(series, lw=0.6); ax[0].set_yscale("symlog", linthresh=10)
ax[0].set_title("Bandwidth over time (log-ish y axis)"); ax[0].set_xlabel("time step"); ax[0].set_ylabel("Mbit/s")
pos = series[series > 0]
ax[1].hist(pos, bins=np.logspace(np.log10(max(pos.min(), 1e-2)), np.log10(pos.max()), 60))
ax[1].set_xscale("log"); ax[1].set_title("Distribution (log x axis)"); ax[1].set_xlabel("Mbit/s")
plt.tight_layout(); plt.show()

s = pd.Series(series)
print(f"median {s.median():.2f} | 95th pct {s.quantile(.95):.1f} | 99th pct {s.quantile(.99):.1f} | max {s.max():.1f}")
print(f"share of readings >= {HIGH_MBPS:g} Mbit/s: {(s >= HIGH_MBPS).mean():.1%}")
print(f"zeros: {(s == 0).sum()}")

If the histogram shows two separate humps, the file mixes two kinds of conditions (for example different links or
units). It is worth finding out why, because the spike readings dominate RMSE. The notebook reports both groups separately below.

## 3. Split in time order and prepare inputs
First 70% train, next 15% validation, last 15% test. Scaling is fitted on the training part only.

In [ ]:
class Scaler:
    """log1p then z-score; fitted on training data only."""
    def fit(self, x):
        z = np.log1p(x); self.mu, self.sd = float(z.mean()), float(z.std() + 1e-8); return self
    def fwd(self, x): return (np.log1p(np.clip(x, 0, None)) - self.mu) / self.sd
    def inv(self, z): return np.clip(np.expm1(np.asarray(z) * self.sd + self.mu), 0, None)

n = len(series)
I1, I2 = int(0.70 * n), int(0.85 * n)
scaler = Scaler().fit(series[:I1])

stats = pd.DataFrame({
    "readings": [I1, I2 - I1, n - I2],
    "mean": [series[:I1].mean(), series[I1:I2].mean(), series[I2:].mean()],
    "median": [np.median(series[:I1]), np.median(series[I1:I2]), np.median(series[I2:])],
    "share >= HIGH": [(a >= HIGH_MBPS).mean() for a in (series[:I1], series[I1:I2], series[I2:])],
}, index=["train", "val", "test"]).round(3)
print(stats.to_string())
print("\nIf these rows look very different, test error will partly reflect a change in conditions, not model quality.")

In [ ]:
def make_features(x_raw):
    """Per-timestep inputs: scaled level, change, rolling mean and std (last 5). Uses past values only."""
    z = scaler.fwd(x_raw).astype(np.float32)
    dz = np.diff(z, prepend=z[0])
    s = pd.Series(z)
    m5 = s.rolling(5, min_periods=1).mean().to_numpy()
    sd5 = s.rolling(5, min_periods=1).std().fillna(0).to_numpy()
    return z, np.stack([z, dz, m5, sd5], axis=1).astype(np.float32)

def prepare(window, horizon):
    z, F = make_features(series)
    T = np.arange(window, n - horizon + 1)                    # index of first target
    X = F[T[:, None] + np.arange(-window, 0)[None, :]]        # (N, window, 4)
    Y = z[T[:, None] + np.arange(horizon)[None, :]]           # (N, horizon) scaled targets
    last_z = X[:, -1, 0]                                      # last observed level
    raw_true = series[T[:, None] + np.arange(horizon)[None, :]]
    base_last = np.repeat(series[T - 1][:, None], horizon, axis=1)
    base_ma5 = np.repeat(np.stack([series[t - 5:t].mean() for t in T])[:, None], horizon, axis=1)
    masks = {  # targets of a split lie strictly inside that split
        "train": T + horizon <= I1,
        "val":   (T >= I1) & (T + horizon <= I2),
        "test":  T >= I2,
    }
    out = {}
    for k, m in masks.items():
        out[k] = dict(X=X[m], Y=(Y[m] - last_z[m, None]).astype(np.float32),  # model predicts CHANGE vs last value
                      last_z=last_z[m], raw_true=raw_true[m], last=base_last[m], ma5=base_ma5[m])
    return out

def metrics(pred, true):
    err = np.abs(pred - true); high = true >= HIGH_MBPS
    return {"MAE": float(err.mean()), "RMSE": float(np.sqrt(((pred - true) ** 2).mean())),
            "MAE_normal": float(err[~high].mean()) if (~high).any() else float("nan"),
            "MAE_spike":  float(err[high].mean())  if high.any()  else float("nan"),
            "MAE_per_step": err.mean(0).round(3).tolist()}

d = prepare(WINDOWS[1], HORIZON)
print({k: v["X"].shape for k, v in d.items()})
print("Baselines on test:")
for name in ("last", "ma5"):
    m = metrics(d["test"][name], d["test"]["raw_true"])
    print(f"  {name:5s} MAE {m['MAE']:.3f}  RMSE {m['RMSE']:.3f}  MAE normal {m['MAE_normal']:.3f}  MAE spike {m['MAE_spike']:.3f}")

## 4. The model

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

class LSTMForecaster(nn.Module):
    def __init__(self, n_feat, hidden, layers, dropout, horizon):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, horizon))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])

@torch.no_grad()
def predict_delta(model, X, batch=1024):
    model.eval(); outs = []
    for i in range(0, len(X), batch):
        outs.append(model(torch.from_numpy(X[i:i + batch]).to(device)).cpu().numpy())
    return np.concatenate(outs)

def to_raw(delta, last_z):
    return scaler.inv(last_z[:, None] + delta)

def train_one(window, horizon=HORIZON, seed=SEED, verbose=True):
    seed_all(seed)
    data = prepare(window, horizon)
    tr, va, te = data["train"], data["val"], data["test"]
    model = LSTMForecaster(tr["X"].shape[2], HIDDEN, LAYERS, DROPOUT, horizon).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    loss_fn = nn.L1Loss()
    loader = DataLoader(TensorDataset(torch.from_numpy(tr["X"]), torch.from_numpy(tr["Y"])), batch_size=BATCH, shuffle=True)
    best, best_state, bad, hist = float("inf"), None, 0, []
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); tot = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(xb)
        tr_loss = tot / len(loader.dataset)
        val_loss = float(np.abs(predict_delta(model, va["X"]) - va["Y"]).mean())
        sched.step(val_loss); hist.append((tr_loss, val_loss))
        if verbose: print(f"  epoch {ep:3d}  train {tr_loss:.4f}  val {val_loss:.4f}")
        if val_loss < best - 1e-5: best, best_state, bad = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                if verbose: print("  early stopping")
                break
    model.load_state_dict(best_state)
    res = {"window": window, "epochs_run": len(hist), "history": hist, "model": model}
    for split in ("val", "test"):
        raw_pred = to_raw(predict_delta(model, data[split]["X"]), data[split]["last_z"])
        res[split] = metrics(raw_pred, data[split]["raw_true"]); res[split + "_pred"] = raw_pred
    res["data"] = data
    return res

## 5. Compare window sizes
Each window length is trained once. The best one is chosen using **validation** MAE only. The test set is looked at once, afterwards.

In [ ]:
results = {}
for w in WINDOWS:
    print(f"Training window={w}")
    results[w] = train_one(w, verbose=False)
    r = results[w]
    print(f"  epochs run {r['epochs_run']} | val MAE {r['val']['MAE']:.3f} | test MAE {r['test']['MAE']:.3f}")

table = pd.DataFrame([{"window": w, "epochs run": r["epochs_run"], "val MAE": r["val"]["MAE"],
                       "test MAE": r["test"]["MAE"], "test RMSE": r["test"]["RMSE"],
                       "test MAE normal": r["test"]["MAE_normal"], "test MAE spike": r["test"]["MAE_spike"]}
                      for w, r in results.items()]).round(3)
display(table)

BEST_W = min(results, key=lambda w: results[w]["val"]["MAE"])
print("Best window by validation MAE:", BEST_W)

## 6. Final results on the test set

In [ ]:
best = results[BEST_W]; td = best["data"]["test"]
rows = {"LSTM (this notebook)": best["test"],
        "Last value": metrics(td["last"], td["raw_true"]),
        "Moving average (5)": metrics(td["ma5"], td["raw_true"])}
final = pd.DataFrame({k: {m: v[m] for m in ("MAE", "RMSE", "MAE_normal", "MAE_spike")} for k, v in rows.items()}).T.round(3)
display(final)

gain = 1 - rows["LSTM (this notebook)"]["MAE"] / rows["Last value"]["MAE"]
print(f"MAE improvement over 'last value': {gain:.1%}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
h, v = zip(*best["history"]); ax[0].plot(h, label="train"); ax[0].plot(v, label="validation")
ax[0].set_title("Training curve (L1 loss, scaled)"); ax[0].set_xlabel("epoch"); ax[0].legend()

steps = np.arange(1, HORIZON + 1); wd = 0.27
for i, (k, v) in enumerate(rows.items()):
    ax[1].bar(steps + (i - 1) * wd, v["MAE_per_step"], wd, label=k)
ax[1].set_title("MAE by prediction step (lower is better)"); ax[1].set_xlabel("steps ahead"); ax[1].set_ylabel("Mbit/s"); ax[1].legend(fontsize=8)

k = min(300, len(td["raw_true"]))
ax[2].plot(td["raw_true"][:k, 0], label="actual", lw=1); ax[2].plot(best["test_pred"][:k, 0], label="LSTM", lw=1)
ax[2].set_yscale("symlog", linthresh=10); ax[2].set_title("Next-step prediction, first test readings"); ax[2].set_xlabel("test sample"); ax[2].legend()
plt.tight_layout(); plt.show()

## 7. Save the model and use it

In [ ]:
torch.save({"state_dict": best["model"].state_dict(), "window": BEST_W, "horizon": HORIZON,
            "hidden": HIDDEN, "layers": LAYERS, "dropout": DROPOUT,
            "scaler_mu": scaler.mu, "scaler_sd": scaler.sd}, "bw_lstm_final.pt")
pd.DataFrame(best["test_pred"], columns=[f"pred_t+{i}" for i in range(1, HORIZON + 1)]).to_csv("test_predictions.csv", index=False)
print("Saved bw_lstm_final.pt and test_predictions.csv")

def forecast(recent_mbps):
    """recent_mbps: the latest readings (at least BEST_W + 5), oldest first. Returns the next HORIZON predictions in Mbit/s."""
    x = np.asarray(recent_mbps, dtype=np.float32)
    assert len(x) >= BEST_W + 5, f"need at least {BEST_W + 5} readings"
    z = scaler.fwd(x)
    dz = np.diff(z, prepend=z[0]); s = pd.Series(z)
    F = np.stack([z, dz, s.rolling(5, min_periods=1).mean().to_numpy(),
                  s.rolling(5, min_periods=1).std().fillna(0).to_numpy()], axis=1).astype(np.float32)
    window = F[-BEST_W:][None]
    return to_raw(predict_delta(best["model"], window), np.array([z[-1]]))[0]

print("Last readings:", series[-5:].round(2))
print("Next", HORIZON, "predicted (Mbit/s):", forecast(series[-(BEST_W + 5):]).round(2))

## Notes
- **Planning use:** to size capacity conservatively you usually want an upper estimate, not the average. A natural next step is quantile outputs (e.g. P10/P50/P90).
- **Spike readings** have far higher error than normal ones. Understanding what produces them (different link? unit change? real bursts?) will help more than any extra tuning.
- **Single trace caveat:** all results come from one time series and one chronological split. Running 3+ seeds, or testing on a different trace, would show how stable the numbers are.